<a href="https://colab.research.google.com/github/amondar/ml00/blob/master/2%20-%20Datasets%20e%20data%20preprocessing/Operare%20su%20variabili%20categoriche.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Operare su dati qualitativi

All'interno di un dataset strutturato puoi trovare due tipologie di dati
<ul>
    <li>**Variabili quantitative continue:** numeri che indicano una quantità</li>
    <li>**Variabili qualiative ordinate (ordinali):** numeri o stringhe che rappresentano delle classi che possono essere ordinate</li>
    <li>**Variabili qualitative sconnesse (nominali):** numeri e stringhe rappresentanti classi che non hanno un ordine</li>
</ul>

<img src="https://github.com/ProfAI/ml00/blob/master/2%20-%20Datasets%20e%20data%20preprocessing/res/variables.png?raw=1" width="350px">

Le variabili qualitative possono essere rappresentate anche da stringhe, in questo caso bisogna codificarle all'interno di numeri per poterle usare come input per un algoritmo di machine learning.
Cominciamo caricando il dataset di esempio, questo contiene un elenco di maglie con le seguenti caratteristiche: prezzo, taglie e colore.

In [1]:
import pandas as pd

shirts = pd.read_csv("https://raw.githubusercontent.com/ProfAI/ml00/master/2%20-%20Datasets%20e%20data%20preprocessing/data/shirts.csv",index_col=0)
shirts.head()

,taglia,colore,prezzo
0,S,bianco,4.99
1,M,bianco,19.99
2,XL,bianco,12.49
3,XL,bianco,14.99
4,S,bianco,14.99


**NOTA BENE**<br>
In inglese le variabili qualitative sono conosciute come *categorical variables (ordinal e nominal)*.

## Mapping delle caratteristiche ordinali
Le variabili qualitative ordinate (ordinali) possono essere ordinate pertanto è possibile rappresentarle come una lista di numeri che rappresentano la posizione dell'elemento all'interno della lista ordinata.
Una funzione non può stimare la posizione di un elemento nella lista, quindi bisogna definirla manualmente con un dizionario.

### Numpy e Scikit-learn
Utilizzando il metodo vectorize di numpy possiamo creare una funzione ottimizzata per operare sugli array, questa funzione ci permetterà di sostituire i labels con i numeri corrispondenti definiti nel dizionario.

In [2]:
import numpy as np

shirts = pd.read_csv("https://raw.githubusercontent.com/ProfAI/ml00/master/2%20-%20Datasets%20e%20data%20preprocessing/data/shirts.csv",index_col=0)
X = shirts.values

size_mapping = {"S":0,"M":1,"L":2,"XL":3} #dizionario che ordina le misure
fmap = np.vectorize(lambda t:size_mapping[t])
X[:,0] = fmap(X[:,0])
X[:5]

array([[0, 'bianco', 4.99],
       [1, 'bianco', 19.99],
       [3, 'bianco', 12.49],
       [3, 'bianco', 14.99],
       [0, 'bianco', 14.99]], dtype=object)

### Pandas
Per sostituire labels con i numeri nel DataFrame possiamo usare il metodo map.

In [3]:
size_mapping = {"S":0,"M":1,"L":2,"XL":3} #dizionario che ordina le misure
shirts["taglia"] = shirts["taglia"].map(size_mapping) #mappiamo la misura con il numero corrispondente
shirts.head()

,taglia,colore,prezzo
0,0,bianco,4.99
1,1,bianco,19.99
2,3,bianco,12.49
3,3,bianco,14.99
4,0,bianco,14.99


# One-hot encoding
Le variabili qualitative sconnesse (nominali) non possono essere mappate come le ordinali, poiché non presentano un ordine interno e quindi è sbagliato assegnare un numero univoco ad ognuna.
Il metodo utilizzato in questo caso è il One-hot encoding, e consiste nel creare una nuova colonna per ogni classe che assumerà il valore 1 se l'esempio appartiene alla classe, 0 nell'altro caso, queste variabili sono chiamate **variabili di comodo (dummy variables)**
<img src="https://github.com/ProfAI/ml00/blob/master/2%20-%20Datasets%20e%20data%20preprocessing/res/onehot.png?raw=1" width="500px">

### Numpy e Scikit-learn
Se il nostro dataset è un array numpy possiamo utilizzare le classi di scikit-learn LabelEncoder e OneHotEncoder

In [4]:
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder

X = shirts.values # Otteniamo l'array numpy corrispondente al DataFrame

le = LabelEncoder() # Per prima cosa mappiamo le classi in numeri
X[:,1] = le.fit_transform(X[:,1]) # eseguiamo il mapping solo sulla colonna del colore (X[:,1])
enc = OneHotEncoder(categorical_features=[1]) # Dopo eseguiamo il one hot encoding,
                                              # specifichiamo di eseguirlo sulla colonna del colore con l'attributo categorical_features
X_sparse = enc.fit_transform(X) # Il risultato sarà una matrice sparsa
X = X_sparse.toarray() # riconvertiamo la matrice sparsa in un array numpy
X[:5] #le prime tre colonne rappresentano la classe-colore (bianco, rosso e verde), la quarta la taglia e la quinta il prezzo

TypeError: OneHotEncoder.__init__() got an unexpected keyword argument 'categorical_features'

### UPDATE Aprile 2019
Nella versione 0.20 di scikit-learn il parametro *caterogical_features* della classe *OneHotEncoder* è stato deprecato e verrà rimosso nella versione 0.22,  inoltre non è più necessario usare il LabelEncoder prima del OneHotEncoder.
Possiamo usare la classe [ColumnTransformer](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html) che applica le trasformazioni solo a determiante colonne. Passiamo al ColumnTransformer una lista di tuple con dentro:
 - un alias per la trasformazione scelto da noi
 - La trasformazione da applicare
 - Una lista con gli indici delle colonne a cui applicare la trasformazione


Utilizzando il parametro *remainder* possiamo definire come comportarci con le colonne che non sono state trasformate, il valore di default è "drop" che le rimuove, impostandolo a "passthrough" le aggiungeremo alla matrice senza applicare alcuna trasformazione.

In [5]:
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer


X = shirts.values
transf = ColumnTransformer([('ohe', OneHotEncoder(), [1])], remainder="passthrough")

X = transf.fit_transform(X)

X[:5]

array([[1.0, 0.0, 0.0, 0, 4.99],
       [1.0, 0.0, 0.0, 1, 19.99],
       [1.0, 0.0, 0.0, 3, 12.49],
       [1.0, 0.0, 0.0, 3, 14.99],
       [1.0, 0.0, 0.0, 0, 14.99]], dtype=object)

### Pandas
Se invece vogliamo eseguire il one-hot encoding su un Dataframe possiamo semplicemente usare il metodo pandas get_dummies

In [6]:
shirts = pd.get_dummies(shirts,columns=["colore"])
shirts.head()

,taglia,prezzo,colore_bianco,colore_rosso,colore_verde
0,0,4.99,True,False,False
1,1,19.99,True,False,False
2,3,12.49,True,False,False
3,3,14.99,True,False,False
4,0,14.99,True,False,False
